In [1]:
import arabic_reshaper
from tensorflow.keras.preprocessing.text import Tokenizer
import pandas as pd
import numpy as np


def input_sim1(input_word):
    input_word = arabic_reshaper.reshape(input_word)

    t = Tokenizer()

    fit_text = "ء أ ـأ ؤ ـؤ إ ـإ ئ ـئ ـئـ ئـ ا ـا ب ـب ـبـ بـ ت ـت ـتـ تـ ة ـة ث ـث ـثـ ثـ ج ـج ـجـ جـ ح ـح ـحـ حـ خ ـخ ـخـ خـ د ـد ذ ـذ ر ـر ز ـز س ـس ـسـ سـ ش ـش ـشـ شـ ص ـص ـصـ صـ ض ـض ـضـ ضـ ط ـط ـطـ طـ ظ ـظ ـظـ ظـ ع ـع ـعـ عـ غ ـغ ـغـ غـ ف ـف ـفـ فـ ق ـق ـقـ قـ ك ـك ـكـ كـ ل ـل ـلـ لـ م ـم ـمـ مـ ن ـن ـنـ نـ ه ـه ـهـ هـ و ـو ى ـى ي ـي ـيـ يـ"

    fit_text = arabic_reshaper.reshape(fit_text)
    t.fit_on_texts(fit_text)

    sequences = t.texts_to_sequences(input_word)

    flattened = [
        value
        for sequence in sequences
        for value in sequence
    ]

    identity = []

    for key in t.word_index.values():
        if key in flattened:
            identity.append("1")
        else:
            identity.append("0")

    return identity, flattened


def encode_allog(word_list, output_file):
    df_word = pd.DataFrame({
        "word": word_list
    })

    allograph_rows = []
    position_rows = []

    for word in word_list:
        identity, position = input_sim1(word)

        allograph_rows.append(identity)
        position_rows.append(position)

    df_allog = pd.DataFrame(allograph_rows)
    df_position = pd.DataFrame(position_rows)

    # Pad shorter position rows with zeros
    df_position.fillna(0, inplace=True)

    # Keep the original string representation for binary allograph values
    df_allog.fillna("0", inplace=True)

    df_allog.columns = range(1, len(df_allog.columns) + 1)
    df_position.columns = range(1, len(df_position.columns) + 1)

    df_word.index = np.arange(1, len(df_word) + 1)
    df_allog.index = np.arange(1, len(df_allog) + 1)
    df_position.index = np.arange(1, len(df_position) + 1)

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df_word.to_excel(
            writer,
            sheet_name="word",
            index=True
        )

        df_allog.to_excel(
            writer,
            sheet_name="allog",
            index=True
        )

        df_position.to_excel(
            writer,
            sheet_name="position",
            index=True
        )

    print(f"Saved to {output_file}")

In [2]:
words= pd.read_excel("PONG/filtered_words/data/words.xlsx")['word'].to_list()
tl_words= pd.read_excel("PONG/filtered_words/data/TL_words.xlsx")['TL_word'].to_list()
dl_words= pd.read_excel("PONG/filtered_words/data/DL_words.xlsx")['DL_word'].to_list()
unr_words= pd.read_excel("PONG/filtered_words/data/unr_words.xlsx")['unr_word'].to_list()

In [3]:
encode_allog(words, 'PONG/filtered_words/words_allog_encodings.xlsx')
encode_allog(tl_words, 'PONG/filtered_words/TL_words_allog_encodings.xlsx')
encode_allog(dl_words, 'PONG/filtered_words/DL_words_allog_encodings.xlsx')
encode_allog(unr_words, 'PONG/filtered_words/unr_words_allog_encodings.xlsx')

Saved to PONG/filtered_words/words_allog_encodings.xlsx
Saved to PONG/filtered_words/TL_words_allog_encodings.xlsx
Saved to PONG/filtered_words/DL_words_allog_encodings.xlsx
Saved to PONG/filtered_words/unr_words_allog_encodings.xlsx
